# ReHA-Net baseline v2 on Colab (GPU)
Runtime > Change runtime type > **T4 GPU**. Drive layout: `MyDrive/reha_baseline_v2/{code,data,results}`.
Cell 4 trains AND re-scores AND plots AND packs a zip. Cells 6-9 only re-run those parts later, or compare seeds.

In [ ]:
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))
# must list one GPU. If empty: Runtime > Change runtime type > T4 GPU, then re-run from here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CODE='/content/drive/MyDrive/reha_baseline_v2/code'; DATA='/content/drive/MyDrive/reha_baseline_v2/data/'; RES='/content/drive/MyDrive/reha_baseline_v2/results'
%cd {CODE}
!ls

In [ ]:
# 3. model check (expect 2,171,921 parameters)
!python main_reha.py --arch v2 --summary | tail -4

In [ ]:
# 4. THE RUN.  Name must match the seed (base_seed0 / base_seed1 / base_seed2).  Re-running the same cell RESUMES after a disconnect.
SEED = 0
NAME = "base_seed%d" % SEED
!python reha_all.py --data-dir {DATA} --result-dir {RES} --name {NAME} --set seed={SEED}

In [ ]:
# 5. OPTIONAL (costs another GPU run): EOG-only training, to compare with the paper's EOG row.  The mixed run already reports an EOG-only SCORE.
!python reha_all.py --data-dir {DATA} --result-dir {RES} --name base_eog_seed0 --set seed=0 noise_type=EOG

In [ ]:
# 6. Re-score a finished run at other SNR settings (inference only; cell 4 already does this once)
RUN = RES + "/" + NAME + "/1"
!python eval_reha.py {RUN} --data-dir {DATA}
!python eval_reha.py {RUN} --snr-convention standard --data-dir {DATA}
!python eval_reha.py {RUN} --snr-low -10 --snr-high 0 --data-dir {DATA}
!python eval_reha.py {RUN} --snr-convention standard --snr-low 10 --snr-high 40 --data-dir {DATA}

In [ ]:
# 7. PLOTS for one run (cell 4 already makes them; this re-makes them).  27 generic plots, then the Phase 2 ones (EOG/EMG, gain, band bias).
RUN = RES + "/" + NAME + "/1"
!python plot_results.py {RUN} --snr-convention repo
!python plots_phase2.py {RUN}

In [ ]:
# 8. Seeds side by side: numbers + mean/range (list every finished run), then plots of them together
RUNS = [RES + "/base_seed0/1", RES + "/base_s2/1"]       # base_s2 = the run that used seed 1 (misnamed); add base_seed2/1 when it exists
LABELS = ["seed 0", "seed 1"]
!python seed_compare.py {" ".join(RUNS)}
!python plots_phase2.py --compare {" ".join(RUNS)} --labels {" ".join('"%s"' % l for l in LABELS)} --out {RES}/compare_seeds

In [ ]:
# 9. How much of the error is amplitude calibration?  (numpy only, seconds)
RUN = RES + "/" + NAME + "/1"
!python gain_check.py {RUN}

Ablation switches (each is one run; add `--set key=value` and a new `--name`): `dropout=0` | `no_residual=True` | `revin=none` | `lambda_cc=0.5` | `arch=v1` | `snr_convention=standard` | `lr_schedule=cosine_restarts` | `lambda_band=0.1`.